In [0]:
%sh
wget -O /Volumes/workspace/steam_reviews/raw_data/steam.csv.bz2 "https://zenodo.org/records/1000885/files/steam.csv.bz2?download=1"


--2026-09-12 02:39:34--  https://zenodo.org/records/1000885/files/steam.csv.bz2?download=1
Resolving zenodo.org (zenodo.org)... 137.138.153.219, 188.184.103.118, 137.138.52.235, ...
Connecting to zenodo.org (zenodo.org)|137.138.153.219|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 505939174 (483M) [text/plain]
Saving to: ‘/Volumes/workspace/steam_reviews/raw_data/steam.csv.bz2’

     0K .......... .......... .......... .......... ..........  0%  172K 47m49s
    50K .......... .......... .......... .......... ..........  0%  365K 35m11s
   100K .......... .......... .......... .......... ..........  0%  365K 30m59s
   150K .......... .......... .......... .......... ..........  0%  365K 28m52s
   200K .......... .......... .......... .......... ..........  0%  355K 27m44s
   250K .......... .......... .......... .......... ..........  0%  364K 26m53s
   300K .......... .......... .......... .......... ..........  0%  364K 26m16s
   350K .......... .......... 

In [0]:
%sh
ls -lh /Volumes/workspace/steam_reviews/raw_data/


total 483M
-rwxrwxrwx 1 spark-68eeff18-12bd-4ee3-8088-4b nogroup 483M Sep 12 02:39 steam.csv.bz2


In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

schema = StructType([
    StructField("game_id", StringType(), True),
    StructField("review_text", StringType(), True),
    StructField("recommended", IntegerType(), True),
    StructField("helpful_votes", IntegerType(), True),
])

df = (
    spark.read
    .option("multiLine", True)
    .option("escape", '"')
    .schema(schema)
    .csv("/Volumes/workspace/steam_reviews/raw_data/steam.csv.bz2")
)

print(f"Rows: {df.count():,}")
df.show(5, truncate=80)


Rows: 6,417,106
+-------+--------------------------------------------------------------------------------+-----------+-------------+
|game_id|                                                                     review_text|recommended|helpful_votes|
+-------+--------------------------------------------------------------------------------+-----------+-------------+
|     10|                                                                 Ruined my life.|          1|            0|
|     10|This will be more of a ''my experience with this game'' type of review, becau...|          1|            1|
|     10|                                                   This game saved my virginity.|          1|            0|
|     10|• Do you like original games? • Do you like games that don't lag? • Do you li...|          1|            0|
|     10|                                        Easy to learn, hard to master.          |          1|            1|
+-------+---------------------------------------

In [0]:
df.write.format("delta").mode("overwrite").saveAsTable("workspace.steam_reviews.reviews_raw")
